In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite"
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    [
        ("system", "You are a helpful AI bot. Your name is {name}."),
        ("human", "categorize the movie review as positive and negative : {movie_name}"),
    ]
)

In [4]:
from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag : Literal["positive" , "negative"]

model_output = model.with_structured_output(llm_schema)

In [5]:
from langchain_core.runnables import RunnableLambda
def pydantic_parser(obj:llm_schema):
    return {"review" : obj.movie_summary_flag }

pydantic_parser_runnable =  RunnableLambda(pydantic_parser)

chain1 = template | model_output | pydantic_parser_runnable

In [ ]:
class poetSchema(BaseModel):
    english_poet:str
    hindi_poet:str


def poetParser(obj:poetSchema):
    return {"english" : obj.branches.englishPoet ,
            "hindi" : obj.branches.hindiPoet
            }

poet_parser_runnable =  RunnableLambda(poetParser)


class hindiSchema(BaseModel):
    hindiPoet:str

def hindiPoetParser(obj:hindiSchema):
    return {"hindi" : obj.hindiPoet}

hindi_poet_parser_runnable =  RunnableLambda(hindiPoetParser)

class englishSchema(BaseModel):
    englishPoet:str

def englishPoetParser(obj:englishSchema):
    return {"english" : obj.englishPoet}

english_poet_parser_runnable =  RunnableLambda(englishPoetParser)

In [12]:
from langchain_core.runnables import RunnableParallel, RunnableBranch

englishTemplate = ChatPromptTemplate.from_messages([
    ("system", "You are a poet writer in english"),
    ("human", "write a poet which has : {review} vibes")])


hindiTemplate = ChatPromptTemplate.from_messages([
    ("system", "You are a peot writer in hindi"),
    ("human", "write a poet which has : {review} vibes")])


hindi_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
english_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

hindi_llm = hindi_llm.with_structured_output(hindiSchema)
english_llm = english_llm.with_structured_output(englishSchema)


hindi_chain = hindiTemplate | hindi_llm | hindi_poet_parser_runnable
english_chain = englishTemplate | english_llm | english_poet_parser_runnable

parallel_chain = chain1 | RunnableParallel(branches = {"englishPoet" : english_chain , "hindiPoet" : hindi_chain})

# movie_name = input("enter your feedback");
# parallel_chain.invoke({"name":"Moview Review Analyzer" , "movie_name": movie_name})


# conditional
# RunnableBranch(
#     (condition1, runnable1),
#     (condition2, runnable2),
#     ...
#     default_runnable
# )

conditional_chain = chain1 | RunnableBranch(
    ((lambda x: x["review"] == 'positive') , english_chain),
    ((lambda x: x["review"] == 'negative') , hindi_chain),
    english_chain
)


movie_name = input("enter your feedback");
conditional_chain.invoke({"name":"Moview Review Analyzer" , "movie_name": movie_name})







{'hindi': 'अंधेरी रात, मन उदास,\nखोया सा मैं, बेसुध आस।\nहर पल में है वीरानगी,\nदिल में बस एक प्यास।\n\nसूरज डूबा, छाए बादल,\nजीवन मेरा, जैसे दलदल।\nहर आशा की लौ बुझ गई,\nबचा है बस एक हलचल।\n\nहर चेहरा लगे अजनबी,\nहर राह लगे सूनसान सी।\nखुद से ही मैं डरता हूँ,\nकैसी ये है बेबसी।'}